## <font color="SteelBlue">Motivação para construção do Projeto:</font>
A empresa onde trabalho possui um sistema de gestão acadêmica, neste sistema existe uma seção "Documentos" onde consta vários PDFs informativos, desde regras, contratos, manuais e regulamentos para consulta dos usuários.

## <font color="Crimson">Problema:</font>
Necessidade de agilidade no processo para encontrar a informação, pois o
método de busca atualmente não é adequado.


## <font color="DarkGreen">Solução: </font>
Construção de um Chat utilizando RAG, para ajudar os usuários a encontrar as informações de forma mais ágil.


## <font color="Indigo">Tecnologia Utilizadas: Ollama, Gradio e  Observabilidade com LangSmith</font>

Este notebook irá guiá-lo através da criação de um sistema d:e Geração Aumentada por Recuperação (RAG) usando:

*   **Ollama com gemma4:12b**: Para o modelo de linguagem grande (LLM).
*   **Documentos PDF**: Localizados no diretório `/content` como base de conhecimento.
*   **Gradio**: Para a interface de chat interativa.

### Visão Geral do Processo:

1.  **Instalação de Dependências**: Instalar bibliotecas Python e configurar o Ollama.
2.  **Carregamento e Processamento de Documentos**: Carregar PDFs, dividi-los em pedaços (chunks) e gerar embeddings.
3.  **Criação de Vector Store**: Armazenar os embeddings em um banco de dados vetorial.
4.  **Configuração da Cadeia RAG**: Criar a lógica para recuperar informações e gerar respostas usando o LLM.
5.  **Interface Gradio**: Integrar tudo em um chat interativo.
6.  **Observabilidade com LangSmith**

## Observalidade com o LangSmith

In [1]:
#instalação da biblioteca lanchain
!pip install langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 26.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


In [2]:
#configurações da chave e do projeto criado no LangSmith

from google.colab import userdata

import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('KEY_LANGSMITH')
os.environ["LANGCHAIN_PROJECT"] = "PROJETO_RAG"  # Opcional

## Passo 1: Instalação do Ollama no Google Colab



In [3]:
# Instala da biblioteca
!pip install -qqq gradio langchain pypdf sentence-transformers chromadb langchain-community langchain-text-splitters > /dev/null

# Instalação do Ollama
# Verifica se está rodando no Google Colab
import os
if 'COLAB_GPU' in os.environ: # Esta variável de ambiente é definida no Colab
    print("Rodando no Google Colab, instalando o Ollama.")
    # Instala zstd, uma dependência para a instalação do Ollama
    !apt-get update -qq > /dev/null
    !apt-get install -qq -y zstd > /dev/null

    !curl -fsSL https://ollama.com/install.sh | sh

    # Verifica a instalação do Ollama
    !ollama --version

else:
    print("Não está rodando no Google Colab. Presumindo que o Ollama já está instalado ou será instalado manualmente.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
Rodando no Google Colab, instalando o Ollama.
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>

## Passo 2: Iniciar o Servidor Ollama e Baixar o Modelo

Precisamos iniciar o servidor Ollama como um processo em segundo plano e, em seguida, baixar o modelo `gemma4:12b`. Isso pode levar alguns minutos, dependendo do tamanho do modelo e da sua conexão com a internet.

In [4]:
# Inicia o Ollama em segundo plano
import subprocess
import time

# Define o caminho completo para o executável do ollama
ollama_executable = "/usr/local/bin/ollama"

# Inicia o servidor Ollama
process = subprocess.Popen([ollama_executable, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Servidor Ollama iniciado em segundo plano.")
time.sleep(10) # Dá um pouco mais de tempo para iniciar

# Baixa o modelo especificado
#model_name = "gemma4:12b"
model_name = "gemma4:12B"

print(f"Baixando o modelo Ollama: {model_name}. Isso pode levar algum tempo...")
subprocess.run([ollama_executable, 'pull', model_name]) # Usa subprocess.run para comando de bloqueio
# !{ollama_executable} pull {model_name} # Alternativa se subprocess.run não funcionar como esperado
print(f"Modelo {model_name} baixado com sucesso.")

Servidor Ollama iniciado em segundo plano.
Baixando o modelo Ollama: gemma4:12B. Isso pode levar algum tempo...
Modelo gemma4:12B baixado com sucesso.


## Passo 3: Preparar Documentos para RAG

Configuração do pipeline de processamento de documentos. Isso inclui:
1.  Criar um diretório `content` (se ele não existir) e instruí-lo a fazer upload de arquivos PDF para lá.
2.  Carregar documentos PDF.
3.  Dividir os documentos em pedaços menores.
4.  Gerar embeddings para esses pedaços.
5.  Armazenar os embeddings em um vetor store Chroma.


In [5]:
!pip install langchain

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters  import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

# Diretório 'content' onde será realizado o upload dos PDFs
content_dir = '/content'
if not os.path.exists(content_dir):
    os.makedirs(content_dir)
    print(f"Diretório criado: {content_dir}. Por favor, faça upload dos seus arquivos PDF aqui.")
else:
    print(f"Diretório '{content_dir}' já existe. Certifique-se de que seus arquivos PDF estão dentro dele.")

# Pega todos os arquivos PDF do diretório content
pdf_files = [os.path.join(content_dir, file) for file in os.listdir(content_dir) if file.endswith('.pdf')]

if not pdf_files:
    print("Nenhum arquivo PDF encontrado no diretório 'content'. Por favor, faça upload de alguns PDFs para continuar.")
else:
    print(f"Encontrados {len(pdf_files)} arquivos PDF: {pdf_files}")

    # Carrega os documentos
    documents = []
    for pdf_file in pdf_files:
        loader = PyPDFLoader(pdf_file)
        documents.extend(loader.load())

    print(f"Carregadas {len(documents)} páginas de documentos.")

    # Divide os documentos em chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1250, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)
    print(f"Dividido em {len(chunks)} chunks.")

    # Cria os embeddings
    # Usando um modelo local Sentence Transformer para embeddings
    embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

    # Cria um store de vetores
    vector_store = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")
    print("Store de vetores criado e persistido em ./chroma_db")

    # Cria um retriever
    retriever = vector_store.as_retriever()
    print("Retriever inicializado.")

/tmp/ipykernel_1321/122868466.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Diretório '/content' já existe. Certifique-se de que seus arquivos PDF estão dentro dele.
Encontrados 9 arquivos PDF: ['/content/CTB_Bolso.pdf', '/content/•Codigo de Transito Brasileiro-_bf427_Codigo de Transito Brasileiro.pdf', '/content/Manual Básico de Segurança no Trânsito - Final.REV02 (1).pdf', '/content/02-infracoes-crimes-penalidades.pdf', '/content/2.-Legislacao-de-Transito_2025.pdf', '/content/cartilha-do-pedestre-e-ciclista.pdf', '/content/tabela_infracao_transito.pdf', '/content/02_INFRACOES_PENALIDADES_OLD.pdf', '/content/Cartilha_-_Educação_para_o_trânsito_ok.pdf']
Carregadas 594 páginas de documentos.
Dividido em 919 chunks.


/tmp/ipykernel_1321/122868466.py:38: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Store de vetores criado e persistido em ./chroma_db
Retriever inicializado.


## Passo 4: Construção da Cadeia RAG

Definição de um prompt para teste do RAG e integrar o `OllamaLLM` com o nosso retriever para criar a cadeia RAG.

In [7]:
!pip install -U langchain-ollama

In [8]:
from langchain_ollama.llms import OllamaLLM

ollama_llm = OllamaLLM(model="gemma4:12B")



In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Definição do Prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda usando exclusivamente o conteúdo fornecido. \n\nContexto:\n{contexto}"),
        ("human", "{query}")
    ]
)




In [10]:
from langchain_core.output_parsers import StrOutputParser

rag_chain = prompt | ollama_llm | StrOutputParser()

In [15]:
# 3. Preparação dos dados do RAG
pergunta = "cuidados que o ciclista deve ter"
trechos = retriever.invoke(pergunta)

# Correção: Garanta as aspas no join para criar a string de contexto
contexto = "\n\n".join(trecho.page_content for trecho in trechos)

# 4. Invocação correta da chain passando o dicionário com as strings
resposta = rag_chain.invoke({
    "query": pergunta,
    "contexto": contexto
})

print(resposta)

Nenhum ciclista (ou qualquer outra pessoa no trânsito) tem o direito de colocar a vida dos outros em risco. Pelo contrário, há o dever de preservar a vida de todos, inclusive a própria.


In [11]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "contexto": RunnablePassthrough() | retriever,
        "query": RunnablePassthrough()
    }
    | prompt
    | ollama_llm
    | StrOutputParser()
)

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Definição do Prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda usando exclusivamente o conteúdo fornecido. \n\nContexto:\n{contexto}"),
        ("human", "{query}")
    ]
)

# 2. Definição da Chain (Garanta que ela combine o prompt, o modelo e o parser)
# Substitua 'modelo_llm' pela sua instância do ChatOpenAI, ChatOllama, etc.
rag_chain = prompt | ollama_llm | StrOutputParser()

# 3. Preparação dos dados do RAG
pergunta = "cuidados que o ciclista deve ter"
trechos = retriever.invoke(pergunta)

# Correção: Garanta as aspas no join para criar a string de contexto
contexto = "\n\n".join(trecho.page_content for trecho in trechos)

# 4. Invocação correta da chain passando o dicionário com as strings
resposta = rag_chain.invoke({
    "query": pergunta,
    "contexto": contexto
})

print(resposta)

Nenhum de nós, seja condutor, pedestre, ciclista ou em outra condição no trânsito tem o direito de colocar a vida dos outros em risco. Ao contrário, temos o dever de preservar a vida de todos, inclusive a nossa.


## Passo 5: Criação do Chat com Interface Gradio

Finalmente, vamos construir uma interface web Gradio para interagir com o nosso sistema RAG. Você pode fazer perguntas e obter respostas com base nos seus documentos PDF.

In [16]:
import gradio as gr

if 'rag_chain' not in locals():
    print("RAG chain não inicializado. Verifique se os passos anteriores foram executados com sucesso.")
else:
    def answer_question(question):
        if not question:
            return "Por favor, faça sua pergunta:"
        try:
            from langchain_core.prompts import ChatPromptTemplate
            from langchain_core.output_parsers import StrOutputParser

            # 1. Definição do Prompt
            prompt = ChatPromptTemplate.from_messages(
               [
                  ("system", "Responda usando exclusivamente o conteúdo fornecido. \n\nContexto:\n{contexto}"),
                  ("human", "{query}")
               ]
                )
            from langchain_core.output_parsers import StrOutputParser

            rag_chain = prompt | ollama_llm | StrOutputParser()

            trechos = retriever.invoke(question)
            contexto = "\n\n".join(trecho.page_content for trecho in trechos)

            answer = rag_chain.invoke({"query":question,"contexto":contexto})
            return answer
        except Exception as e:
            return f"Ocorreu o seguinte erro: {e}"

    # Create Gradio interface
    iface = gr.Interface(
        fn=answer_question,
        inputs=gr.Textbox(lines=2, placeholder="Faça perguntas sobre regras de trânsito."),
        outputs=gr.Textbox(),
        title="RAG Chatbot Regras de Trânsito utilizando Ollama modelo gemma4:12b",
        description="Faça perguntas sobre regras de trânsito."
    )

    print("Carregando Interface...")
    # Launch the Gradio interface
    # The 'share=True' option generates a public, shareable link for 72 hours.
    # In Colab, you'll see a public URL. Click on it to access the interface.
    iface.launch(debug=True, share=True)

Carregando Interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b3587d92f40f758c63.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b3587d92f40f758c63.gradio.live


In [ ]:
from google.colab import drive
drive.mount('/content/drive')